# 01 CDRMIP Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from functools import partial

from cdrmip_extremes import (
    configs, load_pipeline, io_disk, gwl, utils,
    extremes_pipeline, extremes
)


In [2]:
client = Client(memory_limit="6GB",threads_per_worker=1,n_workers=28)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/46395/status,
Dashboard: /proxy/46395/status,Workers: 28
Total threads: 28,Total memory: 156.46 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41669,Workers: 0
Dashboard: /proxy/46395/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36457,Total threads: 1
Dashboard: /proxy/37359/status,Memory: 5.59 GiB
Nanny: tcp://127.0.0.1:38193,


2026-09-30 09:24:06,424 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5f3bcf23a26850dd7699acb632ed13ea initialized by task ('rechunk-merge-rechunk-transfer-60168288ca5931eac537044ca526a635', 0, 0, 0, 47, 0, 0) executed on worker tcp://127.0.0.1:39039
2026-09-30 09:24:59,859 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5f3bcf23a26850dd7699acb632ed13ea deactivated due to stimulus 'task-finished-1790724296.6764798'


In [3]:
config = load_pipeline.LoadConfig(
    experiments=[
        load_pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='day', variable_id='tasmax',
            anomaly_reference='piControl',grid_label=None,
            overshoot=True, crossing_gwls = [1.5], gwl_match_period = (320,340)
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [4]:
pipe = load_pipeline.LoadingPipeline(config)

In [5]:
pipe.load_raw()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'day', 'variable_id': 'tasmax', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-01 12:00:00 -> 1000-12-31 12:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-01 12:00:00 -> 2189-12-31 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-01 12:00:00 -> 2290-12-31 12:00:00
Loading MIROC-ES2L / r1i1p1f2
  1989-01-01 12:00:00 -> 2490-12-31 12:00:00
Loading NorESM2-LM / r1i1p1f1
  0141-01-01 12:00:00 -> 0399-12-31 12:00:00
Loading UKESM1-0-LL / r1i1p1f2
  1990-01-01 12:00:00 -> 2639-12-30 12:00:00


In [6]:
pipe.concatenate_to_parents()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [ACCESS-ESM1-5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 1pctCO2 r1i1p1f2
Concatenating 1pctCO2-cdr -> 1pctCO2 [CNRM-ESM2-1] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CanESM5 1pctCO2-cdr r1i1p2f1 parent
Loading CanESM5 1pctCO2 r1i1p2f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [CanESM5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for MIROC-ES2L 1pctCO2-cdr r1i1p1f2 parent
Loading MIROC-ES2L 1pctCO2 r1i1p1f2
Concatenati

In [7]:
pipe.load_piControl()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 piControl r1i1p1f2
Searching for CanESM5 1pctCO2-cdr r1i1p2f1 parent
Loading CanESM5 piControl r1i1p2f1
Searching for MIROC-ES2L 1pctCO2-cdr r1i1p1f2 parent
Loading MIROC-ES2L piControl r1i1p1f2
Searching for NorESM2-LM 1pctCO2-cdr r1i1p1f1 parent
  Using cached NorESM2-LM 1pctCO2 r1i1p1f1
Searching for UKESM1-0-LL 1pctCO2-cdr r1i1p1f2 parent
Loading UKESM1-0-LL piControl r1i1p1f2


In [8]:
pipe.regrid()

Building regridder for grid (145,) x (192,)
Building regridder for grid (128,) x (256,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (96,) x (144,)
Building regridder for grid (144,) x (192,)


In [9]:
# pipe.load(attribute='regridded_concat')

In [10]:
pipe.regridded_concat

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 21GB
   Dimensions:  (time: 328500, lat: 90, lon: 180)
   Coordinates:
     * time     (time) object 3MB 0000-01-01 12:00:00 ... 0899-12-31 12:00:00
     * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
     * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
   Data variables:
       tasmax   (time, lat, lon) float32 21GB dask.array<chunksize=(357, 90, 180), meta=np.ndarray>
   Attributes:
       regrid_method:  bilinear}},
 'CNRM-ESM2-1': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.Dataset> Size: 8GB
   Dimensions:  (time: 124100, lat: 90, lon: 180)
   Coordinates:
     * time     (time) object 993kB 0000-01-01 12:00:00 ... 0339-12-31 12:00:00
     * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
     * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
   Data variables:
       tasmax   (time, lat, lon) float32 8GB 

## Calculate txx

In [14]:
txx = utils.map_nested(extremes.calc_txx, pipe.regridded_concat)
# 'picontrol' reference (CDRMIP)
pi_ref = utils.map_nested(lambda ds: extremes.calc_txx(ds).mean("year"), pipe.regridded_piControl)
txx_anom = extremes.picontrol_anomaly(txx, pi_ref)

In [15]:
io_disk.save_tree(
    txx_anom,
    configs.PROC_DIR,
    'txx',
    activity_id='CDRMIP',
    category='anomaly'
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_txx_anomaly.nc


## Calc Tx90p

In [11]:
for model, expts in pipe.regridded_concat.items():
    for expt, mems in expts.items():
        for mem, ds in mems.items():
            print(model, expt, mem, ds.time.values[0].calendar)

ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 proleptic_gregorian
CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 standard
CanESM5 1pctCO2-cdr r1i1p2f1 noleap
MIROC-ES2L 1pctCO2-cdr r1i1p1f2 standard
NorESM2-LM 1pctCO2-cdr r1i1p1f1 noleap
UKESM1-0-LL 1pctCO2-cdr r1i1p1f2 360_day


In [12]:
for model, expts in pipe.regridded_concat.items():
    for expt, mems in expts.items():
        for mem, ds in mems.items():
            t = ds["tasmax"].convert_calendar("noleap", align_on="date").time.values[0]
            print(model, type(t).__name__, t.calendar)

ACCESS-ESM1-5 DatetimeNoLeap noleap
CNRM-ESM2-1 DatetimeNoLeap noleap
CanESM5 datetime noleap
MIROC-ES2L DatetimeNoLeap noleap
NorESM2-LM DatetimeNoLeap noleap
UKESM1-0-LL DatetimeNoLeap noleap


In [ ]:
thresh = utils.map_nested(extremes.tx90p_threshold, pipe.regridded_piControl)
concat_340 = utils.map_nested(partial(utils.select_years, lo=0, hi=340), pipe.regridded_concat)
tx90p = utils.map_with_picontrol(extremes.calc_tx90p, concat_340, thresh)

In [ ]:
io_disk.save_tree(
    tx90p,
    configs.PROC_DIR,
    'tx90p',
    activity_id='CDRMIP',
    overwrite=False
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tx90p/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tx90p.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tx90p/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tx90p.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tx90p/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tx90p.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tx90p/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tx90p.nc


## GWL Analysis

In [ ]:
crossing_years = io_disk.load_tree(configs.PROC_DIR, 'crossing_years', 'CDRMIP', as_dataarray=True)

In [20]:
txx_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    txx_anom, crossing_years,
)

In [ ]:
tx90p_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    tx90p, crossing_years,
)

In [21]:
io_disk.save_tree(
    txx_diffs,
    configs.PROC_DIR,
    'txx_gwl_differences',
    activity_id='CDRMIP',
    category=None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_txx_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_txx_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_txx_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx_gwl_differences/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_txx_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx_gwl_differences/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_txx_gwl_differences.nc


In [ ]:
io_disk.save_tree(
    tx90p_diffs,
    configs.PROC_DIR,
    'tx90p_gwl_differences',
    activity_id='CDRMIP',
    category=None
)